# Apple Watch Sleep Dataset — Quality Assessment

Assess the quality of the [Walch et al. (2019)](https://physionet.org/content/sleep-accel/1.0.0/) Apple Watch sleep dataset.

**Data:** 31 subjects with Apple Watch acceleration + heart rate + PSG-labeled sleep stages.

**Goal:** Understand data quality, coverage, sampling rates, and whether this dataset can supplement our own sensor data for model validation. We are NOT training on this data — just checking quality.

## 1. Setup & Load

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict

DATA_DIR = Path("../data/walch_apple_watch/physionet.org/files/sleep-accel/1.0.0")

# Discover subjects
hr_dir = DATA_DIR / "heart_rate"
motion_dir = DATA_DIR / "motion"
labels_dir = DATA_DIR / "labels"
steps_dir = DATA_DIR / "steps"

hr_files = sorted(hr_dir.glob("*_heartrate.txt")) if hr_dir.exists() else []
motion_files = sorted(motion_dir.glob("*_acceleration.txt")) if motion_dir.exists() else []
label_files = sorted(labels_dir.glob("*_labeled_sleep.txt")) if labels_dir.exists() else []

# Extract subject IDs
hr_ids = {f.name.replace("_heartrate.txt", "") for f in hr_files}
motion_ids = {f.name.replace("_acceleration.txt", "") for f in motion_files}
label_ids = {f.name.replace("_labeled_sleep.txt", "") for f in label_files}
all_ids = sorted(hr_ids & motion_ids & label_ids)

print(f"Heart rate files: {len(hr_files)}")
print(f"Motion files: {len(motion_files)}")
print(f"Label files: {len(label_files)}")
print(f"Subjects with all 3 streams: {len(all_ids)}")
print(f"Subject IDs: {all_ids[:5]}..." if len(all_ids) > 5 else f"Subject IDs: {all_ids}")

## 2. Load Functions

In [ ]:
def load_heart_rate(subject_id):
    """Load heart rate data: timestamp (s since PSG start), bpm."""
    path = hr_dir / f"{subject_id}_heartrate.txt"
    df = pd.read_csv(path, header=None, names=["timestamp", "bpm"])
    return df

def load_motion(subject_id):
    """Load acceleration data: timestamp, x, y, z (in g)."""
    path = motion_dir / f"{subject_id}_acceleration.txt"
    df = pd.read_csv(path, header=None, names=["timestamp", "x", "y", "z"])
    return df

def load_labels(subject_id):
    """Load PSG sleep labels: timestamp, stage (0=wake,1=N1,2=N2,3=N3,5=REM)."""
    path = labels_dir / f"{subject_id}_labeled_sleep.txt"
    df = pd.read_csv(path, header=None, names=["timestamp", "stage"])
    return df

# Quick test
if all_ids:
    test_id = all_ids[0]
    hr = load_heart_rate(test_id)
    motion = load_motion(test_id)
    labels = load_labels(test_id)
    print(f"Subject {test_id}:")
    print(f"  HR: {len(hr)} samples, {hr['timestamp'].min():.0f}s - {hr['timestamp'].max():.0f}s")
    print(f"  Motion: {len(motion)} samples, {motion['timestamp'].min():.0f}s - {motion['timestamp'].max():.0f}s")
    print(f"  Labels: {len(labels)} epochs, {labels['timestamp'].min():.0f}s - {labels['timestamp'].max():.0f}s")
    print(f"  HR sample: {hr.head(3).to_string(index=False)}")
    print(f"  Motion sample: {motion.head(3).to_string(index=False)}")
    print(f"  Label sample: {labels.head(3).to_string(index=False)}")

## 3. Per-Subject Quality Report

In [ ]:
STAGE_MAP = {0: "Wake", 1: "N1", 2: "N2", 3: "N3", 5: "REM"}

quality_rows = []

for sid in all_ids:
    hr = load_heart_rate(sid)
    motion = load_motion(sid)
    labels = load_labels(sid)
    
    # Duration
    duration_hr_h = (hr["timestamp"].max() - hr["timestamp"].min()) / 3600
    duration_motion_h = (motion["timestamp"].max() - motion["timestamp"].min()) / 3600
    duration_labels_h = (labels["timestamp"].max() - labels["timestamp"].min()) / 3600
    
    # HR stats
    hr_intervals = hr["timestamp"].diff().dropna()
    hr_median_interval = hr_intervals.median()
    hr_max_gap = hr_intervals.max()
    hr_gaps_gt_30s = (hr_intervals > 30).sum()
    
    # Motion stats
    motion_intervals = motion["timestamp"].diff().dropna()
    motion_median_interval = motion_intervals.median()
    motion_hz = 1.0 / motion_median_interval if motion_median_interval > 0 else 0
    
    # Label stats
    label_intervals = labels["timestamp"].diff().dropna()
    label_epoch_s = label_intervals.median()
    stages_present = labels["stage"].unique()
    has_rem = 5 in stages_present
    rem_epochs = (labels["stage"] == 5).sum()
    rem_pct = rem_epochs / len(labels) * 100 if len(labels) > 0 else 0
    
    # Stage distribution
    stage_dist = labels["stage"].value_counts().to_dict()
    
    # BPM range
    bpm_min = hr["bpm"].min()
    bpm_max = hr["bpm"].max()
    bpm_mean = hr["bpm"].mean()
    
    quality_rows.append({
        "subject": sid,
        "duration_h": round(duration_labels_h, 1),
        "hr_samples": len(hr),
        "hr_median_interval_s": round(hr_median_interval, 1),
        "hr_max_gap_s": round(hr_max_gap, 0),
        "hr_gaps_gt_30s": hr_gaps_gt_30s,
        "bpm_mean": round(bpm_mean, 1),
        "bpm_range": f"{bpm_min:.0f}-{bpm_max:.0f}",
        "motion_samples": len(motion),
        "motion_hz": round(motion_hz, 1),
        "label_epochs": len(labels),
        "label_epoch_s": round(label_epoch_s, 0),
        "has_rem": has_rem,
        "rem_epochs": rem_epochs,
        "rem_pct": round(rem_pct, 1),
        "stages_present": sorted(stages_present),
    })

quality_df = pd.DataFrame(quality_rows)
display(quality_df.drop(columns=["stages_present"]).to_string(index=False))

## 4. Summary Statistics

In [ ]:
print("=" * 60)
print("DATASET QUALITY SUMMARY")
print("=" * 60)
print(f"Total subjects: {len(quality_df)}")
print(f"Subjects with REM: {quality_df['has_rem'].sum()}")
print(f"")
print(f"Session duration (hours):")
print(f"  Mean: {quality_df['duration_h'].mean():.1f}")
print(f"  Min:  {quality_df['duration_h'].min():.1f}")
print(f"  Max:  {quality_df['duration_h'].max():.1f}")
print(f"")
print(f"HR sampling interval (seconds):")
print(f"  Median across subjects: {quality_df['hr_median_interval_s'].median():.1f}")
print(f"  Range: {quality_df['hr_median_interval_s'].min():.1f} - {quality_df['hr_median_interval_s'].max():.1f}")
print(f"")
print(f"HR gaps > 30s:")
print(f"  Mean: {quality_df['hr_gaps_gt_30s'].mean():.1f}")
print(f"  Max:  {quality_df['hr_gaps_gt_30s'].max()}")
print(f"")
print(f"Accelerometer sampling rate (Hz):")
print(f"  Median: {quality_df['motion_hz'].median():.1f}")
print(f"  Range: {quality_df['motion_hz'].min():.1f} - {quality_df['motion_hz'].max():.1f}")
print(f"")
print(f"Sleep label epoch length (seconds):")
print(f"  Median: {quality_df['label_epoch_s'].median():.0f}")
print(f"")
print(f"REM sleep:")
print(f"  Mean epochs per subject: {quality_df['rem_epochs'].mean():.0f}")
print(f"  Mean % of night: {quality_df['rem_pct'].mean():.1f}%")
print(f"  Range: {quality_df['rem_pct'].min():.1f}% - {quality_df['rem_pct'].max():.1f}%")
print(f"")
print(f"Heart rate (BPM):")
print(f"  Mean across subjects: {quality_df['bpm_mean'].mean():.1f}")
print("=" * 60)

## 5. HR Coverage Heatmap

In [ ]:
# For each subject, compute HR coverage per 30-second epoch
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Plot 1: HR sampling intervals distribution
all_hr_intervals = []
for sid in all_ids:
    hr = load_heart_rate(sid)
    intervals = hr["timestamp"].diff().dropna()
    all_hr_intervals.extend(intervals.values)

axes[0].hist(all_hr_intervals, bins=100, range=(0, 60), color="#2980B9", alpha=0.8)
axes[0].set_xlabel("HR Sampling Interval (seconds)")
axes[0].set_ylabel("Count")
axes[0].set_title("HR Sampling Interval Distribution (all subjects)")
axes[0].axvline(5, color="red", linestyle="--", label="5s (Apple Watch typical)")
axes[0].legend()

# Plot 2: Motion sampling intervals distribution
all_motion_intervals = []
for sid in all_ids[:10]:  # Sample 10 subjects (motion is huge)
    motion = load_motion(sid)
    intervals = motion["timestamp"].diff().dropna()
    all_motion_intervals.extend(intervals.values[:10000])  # Cap per subject

axes[1].hist(all_motion_intervals, bins=100, range=(0, 0.5), color="#E74C3C", alpha=0.8)
axes[1].set_xlabel("Accelerometer Sampling Interval (seconds)")
axes[1].set_ylabel("Count")
axes[1].set_title("Accelerometer Sampling Interval Distribution (10 subjects, sampled)")
axes[1].axvline(1/50, color="blue", linestyle="--", label="50Hz")
axes[1].legend()

fig.tight_layout()
fig.savefig("../data/walch_sampling_intervals.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Sleep Stage Distribution

In [ ]:
# Aggregate stage distribution across all subjects
all_stages = []
for sid in all_ids:
    labels = load_labels(sid)
    all_stages.extend(labels["stage"].values)

all_stages = np.array(all_stages)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
stage_counts = {}
for code, name in STAGE_MAP.items():
    stage_counts[name] = (all_stages == code).sum()

colors = ["#F39C12", "#76D7C4", "#76D7C4", "#2980B9", "#E74C3C"]
ax1.bar(stage_counts.keys(), stage_counts.values(), color=colors)
ax1.set_ylabel("Total Epochs")
ax1.set_title("Sleep Stage Distribution (all subjects)")
for i, (name, count) in enumerate(stage_counts.items()):
    pct = count / len(all_stages) * 100
    ax1.text(i, count + 50, f"{pct:.1f}%", ha="center", fontsize=9)

# Per-subject REM percentage
ax2.bar(range(len(quality_df)), quality_df["rem_pct"], color="#E74C3C", alpha=0.8)
ax2.set_xlabel("Subject")
ax2.set_ylabel("REM %")
ax2.set_title("REM Sleep Percentage per Subject")
ax2.axhline(20, color="gray", linestyle="--", label="Typical ~20%")
ax2.legend()

fig.tight_layout()
fig.savefig("../data/walch_stage_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Sample Night Hypnogram + HR Overlay

In [ ]:
# Pick a subject with good REM representation
best_rem_idx = quality_df["rem_epochs"].idxmax()
sample_sid = quality_df.loc[best_rem_idx, "subject"]
print(f"Sample subject: {sample_sid} ({quality_df.loc[best_rem_idx, 'rem_pct']}% REM)")

hr = load_heart_rate(sample_sid)
motion = load_motion(sample_sid)
labels = load_labels(sample_sid)

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# Hypnogram
stage_y_map = {0: 4, 5: 3, 1: 2, 2: 1, 3: 0}  # Wake top, Deep bottom
stage_colors_map = {0: "#F39C12", 1: "#76D7C4", 2: "#76D7C4", 3: "#2980B9", 5: "#E74C3C"}
hours = labels["timestamp"] / 3600
y_vals = labels["stage"].map(stage_y_map)
colors = labels["stage"].map(stage_colors_map)
axes[0].scatter(hours, y_vals, c=colors, s=8, marker="s")
axes[0].set_yticks([0, 1, 2, 3, 4])
axes[0].set_yticklabels(["N3", "N2", "N1", "REM", "Wake"])
axes[0].set_title(f"Hypnogram — Subject {sample_sid}")
axes[0].set_ylabel("Stage")

# Heart rate
hr_hours = hr["timestamp"] / 3600
axes[1].plot(hr_hours, hr["bpm"], color="#2C3E50", linewidth=0.5, alpha=0.7)
axes[1].set_ylabel("Heart Rate (BPM)")
axes[1].set_title("Heart Rate")

# Add REM shading to HR plot
rem_mask = labels["stage"] == 5
label_epoch_s = labels["timestamp"].diff().median()
for _, row in labels[rem_mask].iterrows():
    t_start = row["timestamp"] / 3600
    t_end = (row["timestamp"] + label_epoch_s) / 3600
    axes[1].axvspan(t_start, t_end, alpha=0.15, color="#E74C3C")

# Motion magnitude
# Subsample for plotting
step = max(1, len(motion) // 50000)
m_sub = motion.iloc[::step]
mag = np.sqrt(m_sub["x"]**2 + m_sub["y"]**2 + m_sub["z"]**2)
motion_dev = np.abs(mag - 1.0)
m_hours = m_sub["timestamp"] / 3600
axes[2].plot(m_hours, motion_dev, color="#8E44AD", linewidth=0.3, alpha=0.5)
axes[2].set_ylabel("Motion (|mag - 1g|)")
axes[2].set_xlabel("Hours since PSG start")
axes[2].set_title("Accelerometer Motion")
axes[2].set_ylim(0, 2)

fig.tight_layout()
fig.savefig("../data/walch_sample_night.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Data Gaps & Missing Epochs Analysis

In [ ]:
# For each subject, compute how many 30s epochs have HR data
coverage_rows = []

for sid in all_ids:
    hr = load_heart_rate(sid)
    labels = load_labels(sid)
    
    t_start = labels["timestamp"].min()
    t_end = labels["timestamp"].max()
    epoch_s = labels["timestamp"].diff().median()
    if pd.isna(epoch_s) or epoch_s <= 0:
        epoch_s = 30
    
    n_epochs = len(labels)
    epochs_with_hr = 0
    epochs_with_motion = 0
    
    motion = load_motion(sid)
    
    for _, row in labels.iterrows():
        ep_start = row["timestamp"]
        ep_end = ep_start + epoch_s
        
        hr_in_epoch = hr[(hr["timestamp"] >= ep_start) & (hr["timestamp"] < ep_end)]
        if len(hr_in_epoch) > 0:
            epochs_with_hr += 1
        
        motion_in_epoch = motion[(motion["timestamp"] >= ep_start) & (motion["timestamp"] < ep_end)]
        if len(motion_in_epoch) > 0:
            epochs_with_motion += 1
    
    coverage_rows.append({
        "subject": sid,
        "total_epochs": n_epochs,
        "hr_coverage_pct": round(epochs_with_hr / max(n_epochs, 1) * 100, 1),
        "motion_coverage_pct": round(epochs_with_motion / max(n_epochs, 1) * 100, 1),
    })

coverage_df = pd.DataFrame(coverage_rows)

print("Per-epoch coverage (% of labeled epochs with sensor data):")
print(f"  HR:     mean={coverage_df['hr_coverage_pct'].mean():.1f}%, "
      f"min={coverage_df['hr_coverage_pct'].min():.1f}%, "
      f"max={coverage_df['hr_coverage_pct'].max():.1f}%")
print(f"  Motion: mean={coverage_df['motion_coverage_pct'].mean():.1f}%, "
      f"min={coverage_df['motion_coverage_pct'].min():.1f}%, "
      f"max={coverage_df['motion_coverage_pct'].max():.1f}%")
print()
display(coverage_df.to_string(index=False))

## 9. Comparison to Lullaby Expected Format

In [ ]:
print("=" * 60)
print("WALCH DATASET vs LULLABY EXPECTED FORMAT")
print("=" * 60)
print()
print("Feature comparison:")
print(f"  {'Feature':<30} {'Walch':<15} {'Lullaby':<15}")
print(f"  {'-'*30} {'-'*15} {'-'*15}")
print(f"  {'Heart Rate (BPM)':<30} {'Yes':<15} {'Yes':<15}")
print(f"  {'HRV (SDNN)':<30} {'No':<15} {'Yes':<15}")
print(f"  {'Accelerometer (3-axis)':<30} {'Yes':<15} {'Yes':<15}")
print(f"  {'Sonar breathing rate':<30} {'No':<15} {'Yes':<15}")
print(f"  {'Sonar breathing regularity':<30} {'No':<15} {'Yes':<15}")
print(f"  {'Audio classification':<30} {'No':<15} {'Yes':<15}")
print(f"  {'Audio RMS energy':<30} {'No':<15} {'Yes':<15}")
print(f"  {'Sleep stages (PSG)':<30} {'Yes':<15} {'N/A':<15}")
print(f"  {'Sleep stages (Apple)':<30} {'No':<15} {'Yes':<15}")
print()
print("Sampling rates:")
print(f"  HR:     Walch ~{quality_df['hr_median_interval_s'].median():.0f}s interval  |  Lullaby ~5s")
print(f"  Motion: Walch ~{quality_df['motion_hz'].median():.0f} Hz           |  Lullaby 10 Hz (decimated from 100 Hz)")
print(f"  Labels: Walch {quality_df['label_epoch_s'].median():.0f}s epochs     |  Lullaby 30s epochs (Apple)")
print()
print("Stage label mapping:")
print("  Walch PSG:  Wake(0), N1(1), N2(2), N3(3), REM(5)")
print("  Lullaby:    awake, coreLight, deepSleep, remSleep, inBed")
print("  Mapping:    Wake→awake, N1+N2→coreLight, N3→deepSleep, REM→remSleep")
print()
print("Key differences:")
print("  - Walch has NO HRV, sonar, or audio features (watch-only data)")
print("  - Walch uses PSG labels (gold standard) vs Lullaby uses Apple labels")
print("  - Walch data is from older Apple Watch (Series 2-4 era)")
print("  - Walch has N1/N2 separation; Apple merges them into 'coreLight'")

## 10. Verdict

In [ ]:
print("=" * 60)
print("QUALITY VERDICT")
print("=" * 60)
print()

n_good = len(quality_df[quality_df["has_rem"] & (quality_df["duration_h"] >= 4)])
n_total = len(quality_df)

checks = [
    ("Subjects with all 3 streams", len(all_ids), len(all_ids) >= 20),
    ("Subjects with REM + duration >= 4h", n_good, n_good >= 15),
    ("Mean HR coverage per epoch", f"{coverage_df['hr_coverage_pct'].mean():.0f}%", coverage_df['hr_coverage_pct'].mean() >= 70),
    ("Mean motion coverage per epoch", f"{coverage_df['motion_coverage_pct'].mean():.0f}%", coverage_df['motion_coverage_pct'].mean() >= 70),
    ("PSG sleep stage labels", "Present", True),
    ("HRV data", "Missing", False),
    ("Sonar / audio features", "Missing", False),
]

for name, value, passed in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {name}: {value}")

print()
print("Conclusion:")
print("  This dataset is useful for validating HR + accelerometer → sleep stage")
print("  classification (the watch-side features). However, it CANNOT validate")
print("  sonar breathing, audio features, or HRV — which are the features")
print("  most likely to distinguish REM in our system.")
print()
print("  Recommended use: benchmark the watch-only feature subset of our model")
print("  against PSG ground truth to set a baseline before adding phone features.")